# College Football Winning-Season Classification
CMSE 201 · Fall 2024 · Zachary Roseman

Portfolio edition of the submitted notebook: original modeling code is retained below with condensed narrative and recorded results. The source dataset `cfb23.csv` was not supplied. This notebook has not been rerun. The target is a season win rate greater than 50%, not an individual game outcome.

## Load and preprocess
The original import reported 133 rows, one integer column and 151 object columns. The numeric-only feature selection may therefore use a row index rather than intended football statistics. The original preprocessing is preserved for transparency.

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

file_path = "cfb23.csv"
cfb_data = pd.read_csv(file_path)
cfb_data = cfb_data.drop(index=130)
cfb_data.head(), cfb_data.info()

In [ ]:
# Create binary target variable based on win percentage
cfb_data['Win Target'] = cfb_data['Win-Loss'].apply(
    lambda x: 1 if int(x.split('-')[0]) / sum(map(int, x.split('-'))) > 0.5 else 0
)
# Use all numeric columns as features
X = cfb_data.select_dtypes(include='number').drop(columns=['Win Target'])
y = cfb_data['Win Target']

## Train and evaluate

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)

y_pred_tree = tree_clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

log_reg_acc = accuracy_score(y_test, y_pred_log)
log_reg_report = classification_report(y_test, y_pred_log)

print(f"Logistic Regression Accuracy: {log_reg_acc}")
print("Logistic Regression Report:\n", log_reg_report)

In [ ]:
tree_clf_acc = accuracy_score(y_test, y_pred_tree)
tree_clf_report = classification_report(y_test, y_pred_tree)

print(f"Decision Tree Accuracy: {tree_clf_acc}")
print("Decision Tree Report:\n", tree_clf_report)

## Recorded results (original notebook outputs)

| Model | Accuracy | Winning-season precision | Winning-season recall | Winning-season F1 |
| --- | ---: | ---: | ---: | ---: |
| Logistic regression | 77.5% | 71% | 89% | 79% |
| Decision tree | 72.5% | 70% | 74% | 72% |

Test set: 40 observations. Metrics are recorded, not recomputed.

In [ ]:
results = {
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [log_reg_acc, tree_clf_acc],
}
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(results['Model'], results['Accuracy'], color=['blue', 'green'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.show()

## Limitations and future work
- Validate numeric conversion and exclude identifiers and rank columns; report the actual input columns.
- Season totals are not pregame features; use historical information available before each prediction for a forecasting study.
- Replace row-index exclusions with explicit data validation and compare against a baseline with chronological validation.

Source: [College Football Team Stats Seasons 2013 to 2023](https://www.kaggle.com/datasets/jeffgallini/college-football-team-stats-2019).